# 04 — Model Training and Evaluation

Train LightGBM using log-transformed sales, evaluate on a chronological 16-day recursive validation window, compare with a seasonal baseline, and save predictions, metrics, and feature importance.

In [ ]:
%pip -q install lightgbm joblib pyarrow

In [ ]:
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/retail-demand-forecasting")
FEATURE_PATH = PROJECT / "data/processed/train_features.parquet"
MODEL_DIR = PROJECT / "models"
REPORT_DIR = PROJECT / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        "Feature dataset not found. Run Notebook 03 first."
    )

df = pd.read_parquet(FEATURE_PATH)
df["date"] = pd.to_datetime(df["date"])

lag_cols = [
    "sales_lag_1", "sales_lag_7", "sales_lag_14", "sales_lag_28",
    "sales_rolling_mean_7", "sales_rolling_mean_28"
]

categorical_cols = [
    c for c in ["family", "store_nbr", "city", "state", "type", "cluster"]
    if c in df.columns
]

calendar_cols = [
    "year", "month", "day", "day_of_week", "day_of_year",
    "week_of_year", "is_weekend", "is_month_start", "is_month_end"
]

base_cols = [
    "onpromotion", "dcoilwtico",
    "holiday_event_rows", "transferred_event_rows"
]

holiday_cols = [c for c in df.columns if c.startswith("holiday_type_")]

numeric_cols = [
    c for c in base_cols + calendar_cols + holiday_cols
    if c in df.columns
]

feature_cols = list(dict.fromkeys(
    categorical_cols + numeric_cols + lag_cols
))

missing_features = [c for c in feature_cols if c not in df.columns]
if missing_features:
    raise ValueError(f"Missing features: {missing_features}")

for c in categorical_cols:
    df[c] = df[c].astype("category")

# Chronological split: final 16 days are reserved for validation.
valid_start = df["date"].max().normalize() - pd.Timedelta(days=15)

fit_df = df.loc[df["date"] < valid_start].dropna(
    subset=lag_cols + ["sales"]
).copy()

if fit_df.empty:
    raise ValueError("Training dataset is empty after filtering.")

if (fit_df["sales"] < 0).any():
    raise ValueError("Negative sales require investigation.")

print("Training rows:", f"{len(fit_df):,}")
print("Validation starts:", valid_start.date())
print("Features:", len(feature_cols))
print("Training LightGBM...")

model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

model.fit(
    fit_df[feature_cols],
    np.log1p(fit_df["sales"].to_numpy(dtype="float64")),
    categorical_feature=categorical_cols,
)

joblib.dump(model, MODEL_DIR / "lightgbm_store_sales.joblib")
print("✅ Trained model saved.")

def rmsle(y_true, y_pred):
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    return float(np.sqrt(np.mean(
        (np.log1p(y_true) - np.log1p(y_pred)) ** 2
    )))

# Initialize demand history using only pre-validation actual sales.
history_source = df.loc[
    df["date"] < valid_start,
    ["date", "store_nbr", "family", "sales"]
].sort_values(["store_nbr", "family", "date"])

model_history = {}
baseline_history = {}

for (store, family), group in history_source.groupby(
    ["store_nbr", "family"], sort=False
):
    values = group["sales"].astype(float).tail(28).tolist()
    if len(values) == 28:
        key = (int(store), str(family))
        model_history[key] = values.copy()
        baseline_history[key] = values.copy()

validation_source = df.loc[
    df["date"] >= valid_start
].sort_values(["date", "store_nbr", "family"])

validation_days = sorted(validation_source["date"].unique())
prediction_frames = []

for step, current_date in enumerate(validation_days, start=1):
    day = validation_source.loc[
        validation_source["date"] == current_date
    ].copy()

    lag_values = {c: [] for c in lag_cols}
    baseline_predictions = []
    keys = []

    for store, family in zip(day["store_nbr"], day["family"].astype(str)):
        key = (int(store), str(family))

        if key not in model_history or len(model_history[key]) < 28:
            raise ValueError(f"Insufficient history for {key}")

        h = model_history[key]
        lag_values["sales_lag_1"].append(h[-1])
        lag_values["sales_lag_7"].append(h[-7])
        lag_values["sales_lag_14"].append(h[-14])
        lag_values["sales_lag_28"].append(h[-28])
        lag_values["sales_rolling_mean_7"].append(np.mean(h[-7:]))
        lag_values["sales_rolling_mean_28"].append(np.mean(h[-28:]))

        baseline_predictions.append(max(baseline_history[key][-7], 0.0))
        keys.append(key)

    for col in lag_cols:
        day[col] = lag_values[col]

    forecast = np.maximum(
        np.expm1(model.predict(day[feature_cols])),
        0.0
    )

    baseline_predictions = np.asarray(baseline_predictions, dtype=float)

    result = day[
        ["date", "store_nbr", "family", "sales"]
    ].copy()
    result["baseline_prediction"] = baseline_predictions
    result["model_prediction"] = forecast
    prediction_frames.append(result)

    # Update with predictions only, not actual validation sales.
    for key, prediction, baseline_prediction in zip(
        keys, forecast, baseline_predictions
    ):
        model_history[key] = (
            model_history[key] + [float(prediction)]
        )[-28:]
        baseline_history[key] = (
            baseline_history[key] + [float(baseline_prediction)]
        )[-28:]

    print(f"Validation day {step}/{len(validation_days)} completed.")

results = pd.concat(prediction_frames, ignore_index=True)
results.to_csv(REPORT_DIR / "validation_predictions.csv", index=False)

actual = results["sales"].to_numpy()
baseline_score = rmsle(actual, results["baseline_prediction"])
model_score = rmsle(actual, results["model_prediction"])
mae = float(np.mean(np.abs(actual - results["model_prediction"])))

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

importance.to_csv(REPORT_DIR / "feature_importance.csv", index=False)

metrics = pd.DataFrame([{
    "validation_start": str(valid_start.date()),
    "validation_end": str(results["date"].max().date()),
    "validation_rows": len(results),
    "baseline_rmsle": baseline_score,
    "model_rmsle": model_score,
    "model_mae": mae,
    "improvement_percent": (
        (baseline_score - model_score) / baseline_score * 100
        if baseline_score else np.nan
    ),
}])

metrics.to_csv(REPORT_DIR / "model_metrics.csv", index=False)

print("\n===== VALIDATION REPORT =====")
print(metrics.to_string(index=False))
print("\nTop 15 features:")
print(importance.head(15).to_string(index=False))
print("\n✅ Model and reports saved in:", PROJECT)
